# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Follows `checklist.md`.

Decisions up front:
- **Leakage**: `parts_replaced` is recorded on the work order that *closed* the inspection record, i.e. after the inspection (and its values 2+ always mean failure). Not known at prediction time -> dropped. `asset_tag` is an identifier -> dropped. `inspected_at` is used only for ordering, not as a feature.
- **Split**: the model is meant for future inspections, so the test set is the latest 20% of rows by time. Each asset appears in exactly one row, so no entity overlap is possible.
- **CV**: expanding-window forward splits inside the training rows only.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
from sklearn.base import clone

df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, df.asset_tag.is_unique, df.isna().sum().sum())
print(df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.agg(['mean','size']))

Note: the failure rate varies strongly over time (peak mid-2024, lowest in 2025Q4). The last quarter may also be partly right-censored if failures were only observed up to the data extraction date, which would bias the test rate downward. We keep this in mind when reading the test score.

## 1. Set aside the test set (latest 20% of rows)

In [ ]:
n_test = int(len(df)*0.2)
train, test = df.iloc[:-n_test].copy(), df.iloc[-n_test:].copy()
assert train.inspected_at.max() <= test.inspected_at.min()
assert not set(train.asset_tag) & set(test.asset_tag)
print(len(train), len(test), train.inspected_at.max(), test.inspected_at.min())
FEATURES = ['site','vibration_rms','bearing_temp_c','ambient_temp_c','meter_hours','load_pct']  # no parts_replaced, no ids
NUM = [f for f in FEATURES if f!='site']
X_tr, y_tr = train[FEATURES], train.failed_90d
X_te, y_te = test[FEATURES], test.failed_90d  # not touched until step 5

## 2. Candidate models (named before any scoring)
1. `baseline`: predicts the training prevalence (no features).
2. `logreg`: logistic regression, scaled numerics + one-hot site, C in {0.1, 1, 10}.
3. `hgb`: histogram gradient boosting, max_depth in {2, 4}, learning_rate 0.05, 200 iterations max with early stopping.

Selection metric: mean forward-CV average precision (PR-AUC; the classes are imbalanced). Decision threshold: maximises F1 on pooled out-of-fold CV predictions of the chosen model. All preprocessing sits inside the pipelines, so it is fit within each fold.

In [ ]:
pre = ColumnTransformer([('num', StandardScaler(), NUM), ('site', OneHotEncoder(handle_unknown='ignore'), ['site'])])
cands = {'baseline': Pipeline([('m', DummyClassifier(strategy='prior'))])}
for C_ in [0.1,1,10]:
    cands[f'logreg_C{C_}'] = Pipeline([('pre', pre), ('m', LogisticRegression(C=C_, max_iter=1000))])
for d in [2,4]:
    cands[f'hgb_d{d}'] = Pipeline([('pre', pre), ('m', HistGradientBoostingClassifier(max_depth=d, learning_rate=0.05, max_iter=200, early_stopping=True, random_state=0))])
print(list(cands))

## 3. Forward cross-validation on training rows only

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
rows, oof = [], {}
for name, est in cands.items():
    p = np.full(len(X_tr), np.nan)
    aps, aucs = [], []
    for tr, va in tscv.split(X_tr):
        m = clone(est).fit(X_tr.iloc[tr], y_tr.iloc[tr])
        p[va] = m.predict_proba(X_tr.iloc[va])[:,1]
        aps.append(average_precision_score(y_tr.iloc[va], p[va]))
        aucs.append(roc_auc_score(y_tr.iloc[va], p[va]))
    oof[name] = p
    rows.append((name, np.mean(aps), np.std(aps), np.mean(aucs)))
cv = pd.DataFrame(rows, columns=['model','cv_ap','cv_ap_sd','cv_auc']).sort_values('cv_ap', ascending=False)
cv

## 4. Choose model and threshold on CV only

In [ ]:
best = cv.iloc[0].model
valid = ~np.isnan(oof[best])          # rows that were in some validation fold
yv, pv = y_tr.values[valid], oof[best][valid]
ths = np.linspace(0.02, 0.9, 89)
f1s = [f1_score(yv, pv>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(thr,3), '| CV F1:', round(max(f1s),3))

**Choice recorded before test scoring:** the model named above (top CV average precision) with the F1-maximising out-of-fold threshold printed above, refit on all training rows. Nothing below changes it.

## 5. Refit on all training rows; score the test set once

In [ ]:
final = clone(cands[best]).fit(X_tr, y_tr)
pt = final.predict_proba(X_te)[:,1]
base = clone(cands['baseline']).fit(X_tr, y_tr).predict_proba(X_te)[:,1]
pred = pt >= thr
print('test prevalence      :', round(y_te.mean(),3))
print('ROC AUC              :', round(roc_auc_score(y_te, pt),3))
print('Average precision    :', round(average_precision_score(y_te, pt),3), '(baseline = prevalence', round(average_precision_score(y_te, base),3), ')')
print('At threshold %.3f -> precision %.3f, recall %.3f, F1 %.3f' % (thr, precision_score(y_te,pred), recall_score(y_te,pred), f1_score(y_te,pred)))

## Conclusion
See the test-set numbers above (the headline). Caveats: the test period has a much lower failure rate than the earlier training period (drift, possibly partly right-censoring of recent labels), so the CV-chosen threshold may be miscalibrated for it and precision will be affected. `parts_replaced` would look like a near-perfect predictor but is a post-inspection leak and was excluded deliberately.